# Step 3: Data preprocessing

From the resampled CWRU bearing data (step 2), we perform data selection and FL dataset split, and train-test split

The specs follow:
- https://www.sciencedirect.com/science/article/pii/S0888327021010499
- https://arxiv.org/abs/2407.14625

The resulting data is called the data_split.


In [8]:
# Load the "autoreload" extension so that code can change
%load_ext autoreload
# Always reload modules so that as you change code in src, it gets loaded
%autoreload 2

import os
from copy import deepcopy
import json
import yaml
import re

import numpy as np
import pandas as pd
import scipy
from scipy.signal import ShortTimeFFT
import librosa

import matplotlib.pyplot as plt

from data_selection_split import select_cwru_data

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
source_data_dir = './data_resampled'

## Load metadata

In [4]:
source_metadata_path = os.path.join(source_data_dir, 'metadata.json')
with open(source_metadata_path, 'r') as f:
    metadata = json.load(f)

In [5]:
len(metadata)

153

In [6]:
df_metadata = pd.DataFrame(metadata)

In [7]:
df_metadata.head()

,raw_data_path,cycle_id,fault_label,motor_load_hp,sampling_freq_khz,fault_type,fault_location,fault_diameter,OR_position,rpm,df_length,df_columns,parsed_data_path,resampled_data_path
0,./data/Normal/99_Normal_2.mat,99,N,2,12.0,N,None,None,None,1750.0,121265,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/99.csv,./data_resampled/99.csv
1,./data/Normal/98_Normal_1.mat,98,N,1,12.0,N,None,None,None,1772.0,120975,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/98.csv,./data_resampled/98.csv
2,./data/Normal/97_Normal_0.mat,97,N,0,12.0,N,None,None,None,1797.0,121969,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/97.csv,./data_resampled/97.csv
3,./data/Normal/100_Normal_3.mat,100,N,3,12.0,N,None,None,None,1730.0,121410,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/100.csv,./data_resampled/100.csv
4,./data/12k_Drive_End_Bearing_Fault_Data/B/007/...,121,DE_B,3,12.0,DE,B,007,None,1722.0,121556,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/121.csv,./data_resampled/121.csv


## Data selection for training

    Only use the 12kHz data (before resampling), which includes the normal bearing data

    Only keep the fault diameters 0.007”, 0.014” and 0.021”

    Use all of motor load = 0, 1, 2, 3HP; in those papers they dropped the 0HP data. We include it to increase data volume

    For outer ring faults, the ‘OR_position' of ‘@6’ is used whenever possible, or else '@3’

Use 12kHz data and drop 48kHz data (before resampling)

In [9]:
selected_metadata = select_cwru_data(selected_output_dir='test')

In [16]:
df_metadata = pd.DataFrame(selected_metadata)

## Data split

Following a discussion with Eric, we decided that the following data split strategy into FL clients is more generalisable to a real-life setting:

The dataset is split into FL clients by the motor load (in HP) and fault size (diameter). That is, a given FL client has only bearing data of a single value of motor load (in HP) and fault size (diameter) respectively. We find it reasonable to assume that, in many situations, the bearing in each FL client runs at a fixed load, and has only one fault size

Recall that fault diameters = 0.007”, 0.014” and 0.021”, and motor load = 0, 1, 2, 3HP

If we insist that all clients have all the fault label, then we have 12 clients; we put those 4 healthy cycles in some of those clients of the same motor load randomly

Or, we can consider the case where each FL client only has a subset of the fault label classes (excluding normal)

In [28]:
fl_split_group: list = ["fault_diameter", "motor_load_hp"]
normal_cycles = df_metadata[df_metadata['fault_type'] == 'N']
fault_cycles = df_metadata[df_metadata['fault_type'] != 'N']
fault_cycles = fault_cycles.set_index(fl_split_group).sort_index()

In [33]:
fault_cycles.shape

(68, 13)

In [32]:
fault_cycles.loc[('007', '0')].shape

(6, 13)

In [34]:
count = 0
for ind in fault_cycles.index.unique():
    count += fault_cycles.loc[ind].shape[0]

In [35]:
count

68

In [163]:
df_metadata_selected[df_metadata_selected['selected_data_path'].isnull()]

,raw_data_path,cycle_id,fault_label,motor_load_hp,sampling_freq_khz,fault_type,fault_location,fault_diameter,OR_position,rpm,df_length,df_columns,parsed_data_path,resampled_data_path,selected_data_path


In [114]:
df_metadata_selected

,raw_data_path,cycle_id,fault_label,motor_load_hp,sampling_freq_khz,fault_type,fault_location,fault_diameter,OR_position,rpm,df_length,df_columns,parsed_data_path,resampled_data_path,selected_data_path
0,./data/Normal/99_Normal_2.mat,99,N,2,12.0,N,None,None,None,1750.0,121265,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/99.csv,./data_resampled/99.csv,99.csv
1,./data/Normal/98_Normal_1.mat,98,N,1,12.0,N,None,None,None,1772.0,120975,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/98.csv,./data_resampled/98.csv,98.csv
2,./data/Normal/97_Normal_0.mat,97,N,0,12.0,N,None,None,None,1797.0,121969,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/97.csv,./data_resampled/97.csv,97.csv
3,./data/Normal/100_Normal_3.mat,100,N,3,12.0,N,None,None,None,1730.0,121410,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/100.csv,./data_resampled/100.csv,100.csv
4,./data/12k_Drive_End_Bearing_Fault_Data/B/007/...,121,DE_B,3,12.0,DE,B,007,None,1722.0,121556,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/121.csv,./data_resampled/121.csv,121.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,./data/12k_Fan_End_Bearing_Fault_Data/OR/014/@...,313,FE_OR,0,12.0,FE,OR,014,@6,1797.0,120984,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/313.csv,./data_resampled/313.csv,313.csv
101,./data/12k_Fan_End_Bearing_Fault_Data/OR/021/@...,316,FE_OR,1,12.0,FE,OR,021,@3,1773.0,120617,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/316.csv,./data_resampled/316.csv,316.csv
102,./data/12k_Fan_End_Bearing_Fault_Data/OR/021/@...,317,FE_OR,2,12.0,FE,OR,021,@3,1751.0,120617,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/317.csv,./data_resampled/317.csv,317.csv
103,./data/12k_Fan_End_Bearing_Fault_Data/OR/021/@...,318,FE_OR,3,12.0,FE,OR,021,@3,1728.0,121351,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/318.csv,./data_resampled/318.csv,318.csv


In [110]:
fl_split_group = ["fault_diameter", "motor_load_hp"]
normal_cycles = df_metadata_selected[df_metadata_selected['fault_diameter'].isna()]
fl_groups = df_metadata_selected[fl_split_group].dropna()

In [135]:
fl_groups

,fault_diameter,motor_load_hp
4,007,3
5,007,1
6,007,0
7,007,2
8,014,1
...,...,...
100,014,0
101,021,1
102,021,2
103,021,3


In [139]:
df_metadata_selected1 = df_metadata_selected.dropna(subset=fl_split_group).set_index(fl_split_group)

In [ ]:
fl_df_metadatas = []
for ind in df_metadata_selected1.index:
    fl_df_metadatas.append(df_metadata_selected1.loc[ind].reset_index())

In [151]:
prng = np.random.default_rng(42)

In [153]:
prng.choice(len(fl_df_metadatas), 4, replace=False)

array([ 5, 56, 47, 28])

In [159]:
df_metadata.iloc[-5:]

,raw_data_path,cycle_id,fault_label,motor_load_hp,sampling_freq_khz,fault_type,fault_location,fault_diameter,OR_position,rpm,df_length,df_columns,parsed_data_path,resampled_data_path
148,./data/48k_Drive_End_Bearing_Fault_Data/OR/021...,240,DE_OR,2,12.0,DE,OR,021,@6,1747.0,121990,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/240.csv,./data_resampled/240.csv
149,./data/48k_Drive_End_Bearing_Fault_Data/OR/021...,264,DE_OR,2,12.0,DE,OR,021,@12,1746.0,121701,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/264.csv,./data_resampled/264.csv
150,./data/48k_Drive_End_Bearing_Fault_Data/OR/021...,265,DE_OR,3,12.0,DE,OR,021,@12,1718.0,121556,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/265.csv,./data_resampled/265.csv
151,./data/48k_Drive_End_Bearing_Fault_Data/OR/021...,263,DE_OR,1,12.0,DE,OR,021,@12,1771.0,121556,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/263.csv,./data_resampled/263.csv
152,./data/48k_Drive_End_Bearing_Fault_Data/OR/021...,262,DE_OR,0,12.0,DE,OR,021,@12,1796.0,120506,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/262.csv,./data_resampled/262.csv


In [161]:
df_metadata.iloc[:len(df_metadata)-5 -3]

,raw_data_path,cycle_id,fault_label,motor_load_hp,sampling_freq_khz,fault_type,fault_location,fault_diameter,OR_position,rpm,df_length,df_columns,parsed_data_path,resampled_data_path
0,./data/Normal/99_Normal_2.mat,99,N,2,12.0,N,None,None,None,1750.0,121265,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/99.csv,./data_resampled/99.csv
1,./data/Normal/98_Normal_1.mat,98,N,1,12.0,N,None,None,None,1772.0,120975,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/98.csv,./data_resampled/98.csv
2,./data/Normal/97_Normal_0.mat,97,N,0,12.0,N,None,None,None,1797.0,121969,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/97.csv,./data_resampled/97.csv
3,./data/Normal/100_Normal_3.mat,100,N,3,12.0,N,None,None,None,1730.0,121410,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/100.csv,./data_resampled/100.csv
4,./data/12k_Drive_End_Bearing_Fault_Data/B/007/...,121,DE_B,3,12.0,DE,B,007,None,1722.0,121556,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/121.csv,./data_resampled/121.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140,./data/48k_Drive_End_Bearing_Fault_Data/OR/007...,164,DE_OR,3,12.0,DE,OR,007,@12,1723.0,121120,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/164.csv,./data_resampled/164.csv
141,./data/48k_Drive_End_Bearing_Fault_Data/OR/021...,251,DE_OR,1,12.0,DE,OR,021,@3,1770.0,122426,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/251.csv,./data_resampled/251.csv
142,./data/48k_Drive_End_Bearing_Fault_Data/OR/021...,253,DE_OR,3,12.0,DE,OR,021,@3,1719.0,121120,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/253.csv,./data_resampled/253.csv
143,./data/48k_Drive_End_Bearing_Fault_Data/OR/021...,250,DE_OR,0,12.0,DE,OR,021,@3,1796.0,128663,"[timestamp, DE_time, FE_time, cycle_id]",./data_parsed/250.csv,./data_resampled/250.csv
